In [ ]:
!pip install ftfy
!pip install transformers
!pip install laserembeddings
!pip install tensorflow_addons
!pip install vncorenlp

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
from google.colab import drive
import numpy as np
import pickle
import tensorflow as tf
import random
import tensorflow_addons as tfa
import pandas as pd
import warnings,operator
from tqdm import tqdm
from ftfy import fix_text
import os, re, string

warnings.filterwarnings("ignore")
from transformers import logging
from transformers import AutoTokenizer, AutoModel,AutoConfig, TFAutoModel, XLMRobertaTokenizer,XLMRobertaConfig,TFXLMRobertaModel
logging.set_verbosity_error()

from google.colab import drive
drive.mount('/content/gdrive')
path_train ='/content/gdrive/My Drive/2023_Research/JCSCE Response/Dataset/Train.txt'
path_dev ='/content/gdrive/My Drive/2023_Research/JCSCE Response/Dataset/Dev.txt'
path_test ='/content/gdrive/My Drive/2023_Research/JCSCE Response/Dataset/Test.txt'

!cp "/content/gdrive/MyDrive/2021_Research/ISI 2022/Journal 1/Cate-Senti Classification/EvaluationSystemByFile.py" "EvaluationSystemByFile.py"
#!cp "/content/gdrive/MyDrive/Journal 1/Cate-Senti Classification/EvaluationSystemByFile.py" "EvaluationSystemByFile.py"


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
def set_seeds(seed=1):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    np.random.seed(seed)

def set_global_determinism(seed=1):
    set_seeds(seed=seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'


# Call the above function with seed value
# 1,2,3,4,5
SEED = 42
set_global_determinism(seed=SEED)
tf.keras.utils.set_random_seed(SEED)

In [ ]:
class Review():
    def __init__(self,stt,text,label):
        self.stt = stt
        self.labels = label
        self.orginalText = text

def read_files(filename):
    documents = list()
    count = 0
    with open(filename,'r',encoding='utf8') as file:
        text1 = file.read()
        for item in text1.split('\n'):
            if item != '':
                item = item.strip()
                if count == 0:
                    stt = fix_text(item)
                    count += 1
                elif count == 1:
                    text = fix_text(item)
                    count += 1
                elif count == 2:
                    labels = fix_text(item)
                    count = 0
                    reviewtemp = Review(stt, text, labels)
                    documents.append(reviewtemp)
    return documents

In [ ]:
from vncorenlp import VnCoreNLP
rdrsegmenter = VnCoreNLP("/content/gdrive/MyDrive/2021_Research/Text Classification Research/vncorenlp/VnCoreNLP-1.1.1.jar", annotators="wseg", max_heap_size='-Xmx500m')
#rdrsegmenter = VnCoreNLP("/content/gdrive/MyDrive/vncorenlp/VnCoreNLP-1.1.1.jar", annotators="wseg", max_heap_size='-Xmx500m')


def normalText(sent):
    sent = str(sent).replace('_',' ').replace('/',' trên ')
    sent = re.sub('-{2,}','',sent)
    sent = re.sub('\\s+',' ', sent)
    patPrice = r'([0-9]+k?(\s?-\s?)[0-9]+\s?(k|K))|([0-9]+(.|,)?[0-9]+\s?(triệu|ngàn|trăm|k|K|))|([0-9]+(.[0-9]+)?Ä‘)|([0-9]+k)'
    patHagTag = r'#\s?[aăâbcdđeêghiklmnoôơpqrstuưvxyàằầbcdđèềghìklmnòồờpqrstùừvxỳáắấbcdđéếghíklmnóốớpqrstúứvxýảẳẩbcdđẻểghỉklmnỏổởpqrstủửvxỷạặậbcdđẹệghịklmnọộợpqrstụựvxỵãẵẫbcdđẽễghĩklmnõỗỡpqrstũữvxỹAĂÂBCDĐEÊGHIKLMNOÔƠPQRSTUƯVXYÀẰẦBCDĐÈỀGHÌKLMNÒỒỜPQRSTÙỪVXỲÁẮẤBCDĐÉẾGHÍKLMNÓỐỚPQRSTÚỨVXÝẠẶẬBCDĐẸỆGHỊKLMNỌỘỢPQRSTỤỰVXỴẢẲẨBCDĐẺỂGHỈKLMNỎỔỞPQRSTỦỬVXỶÃẴẪBCDĐẼỄGHĨKLMNÕỖỠPQRSTŨỮVXỸ]+'
    patURL = r"(?:http://|www.)[^\"]+"
    sent = re.sub(patURL,'website',sent)
    sent = re.sub(patHagTag,' hagtag ',sent)
    sent = re.sub(patPrice, ' giá_tiền ', sent)
    sent = re.sub('\.+','.',sent)
    sent = re.sub('(hagtag\\s+)+',' hagtag ',sent)
    sent = re.sub('\\s+',' ',sent)
    return sent


def normalize_elonge_word(sent):
    s_new = ''
    for word in sent.split(' '):
        word_new = ''
        for char in word.strip():
            if char != word_new[-1]:
                word_new += char
    s_new += word_new.strip() + ' '
    return s_new

def tokenizer(text):
    token = rdrsegmenter.tokenize(text)
    token = ' '.join([' '.join(x) for x in token])
    token = token.replace('giá tiền','giá_tiền').replace('Giá tiền','Giá_tiền')
    return token

def deleteIcon(text):
    text = text.lower()
    s = ''
    pattern = r"[a-zA-ZaăâbcdđeêghiklmnoôơpqrstuưvxyàằầbcdđèềghìklmnòồờpqrstùừvxỳáắấbcdđéếghíklmnóốớpqrstúứvxýảẳẩbcdđẻểghỉklmnỏổởpqrstủửvxỷạặậbcdđẹệghịklmnọộợpqrstụựvxỵãẵẫbcdđẽễghĩklmnõỗỡpqrstũữvxỹAĂÂBCDĐEÊGHIKLMNOÔƠPQRSTUƯVXYÀẰẦBCDĐÈỀGHÌKLMNÒỒỜPQRSTÙỪVXỲÁẮẤBCDĐÉẾGHÍKLMNÓỐỚPQRSTÚỨVXÝẠẶẬBCDĐẸỆGHỊKLMNỌỘỢPQRSTỤỰVXỴẢẲẨBCDĐẺỂGHỈKLMNỎỔỞPQRSTỦỬVXỶÃẴẪBCDĐẼỄGHĨKLMNÕỖỠPQRSTŨỮVXỸ,._]"

    for char in text:
        if char !=' ':
            if len(re.findall(pattern, char)) != 0:
                s+=char
            elif char == '_':
                s+=char
        else:
            s+=char
    s = re.sub('\\s+',' ',s)
    return s.strip()

def normalize_elonge_word(sent):
    s_new = ''
    for word in sent.split(' '):
        word_new = ' '
        for char in word.strip():
            if char != word_new[-1]:
                word_new += char
        s_new += word_new.strip() + ' '
    return s_new.strip()

def clean_doc(doc, word_segment=False, lower_case=False):
    for punc in string.punctuation:
        doc = doc.replace(punc,' '+ punc + ' ')
    doc = normalText(doc)
    doc = deleteIcon(doc)
    # Removing multiple whitespaces
    doc = re.sub(r"\?", " \? ", doc)
    # Remove numbers
    doc = re.sub(r"[0-9]+", " num ", doc)
    # Split in tokens
    doc = re.sub('\\s+',' ',doc)
    doc = normalize_elonge_word(doc)
    if lower_case == True:
      doc = doc.lower()

    if word_segment == True:
      doc = rdrsegmenter.tokenize(doc)
      doc = ' '.join([' '.join(x) for x in doc])
      doc = doc.translate(doc.maketrans('', '', string.punctuation.replace("_",""))).replace("giá _ tiền", "giá_tiền").replace("giátiền", "giá_tiền")
    else:
        doc = doc.translate(doc.maketrans('', '', string.punctuation)).replace("giá _ tiền", "giá tiền").replace("giátiền", "giá tiền")

    doc = re.sub('\\s+',' ',doc)
    doc = doc.strip()
    return doc

print(clean_doc("người ta có bạn bè nhìn vui thật", word_segment=True, lower_case=True))
print(clean_doc("giá 45k giá ngon", word_segment=False, lower_case=True))

người ta có bạn_bè nhìn vui thật
giá giá tiền giá ngon


In [ ]:
def read_data_train():
    data  = []
    for review in read_files(path_train):
        data.append(review)
    print(len(data))
    return data

def read_data_dev():
    data  = []
    for review in read_files(path_dev):
        data.append(review)
    print(len(data))
    return data


def read_data_test():
    data = []
    for review in read_files(path_test):
        data.append(review)
    print(len(data))
    return data

In [ ]:
def to_category_vector(label):
    vector = np.zeros(6).astype(np.float64)
    if label.strip() != '':
        if 'very_positive' in label:
            vector[1] = 1.0
        elif 'positive' in label:
            vector[2] = 1.0
        elif 'neutral' in label:
            vector[3] = 1.0
        elif 'negative' in label:
            vector[4] = 1.0
        elif 'very_negative' in label:
            vector[5] = 1.0
        else:
            vector[0] = 1.0
    else:
        vector[0] = 1.0
    return vector

# Hàm chuyển input nhãn thành vector cho mỗi nhãn aspect
import re
def create_output_aspect_polarity(labels,categories,list_output):
    for i,category in enumerate(categories):
        if category in labels:
            try:
                output = re.findall('('+category+',\s(very_positive|positive|negative|neutral|very_negative))',labels)[0][0]
            except:
                output = ''
            list_output[i].append(to_category_vector(output))
        else:
            list_output[i].append(to_category_vector(''))
    return list_output

In [ ]:
listLabel = 'QUALITY,LOCATION,PRICES,AMBIENCE,SERVICE,STYLE_OPTIONS,MISCELLANEOUS'
categories = listLabel.split(',')

x_train_segment = []
x_train_nosegment = []

output_train1 = []
output_train2 = []
output_train3 = []
output_train4 = []
output_train5 = []
output_train6 = []
output_train7 = []


list_output_train = list()
list_output_train.append(output_train1)
list_output_train.append(output_train2)
list_output_train.append(output_train3)
list_output_train.append(output_train4)
list_output_train.append(output_train5)
list_output_train.append(output_train6)
list_output_train.append(output_train7)


for review in read_data_train():
    x_train_segment.append(clean_doc(review.orginalText,True,False))
    x_train_nosegment.append(clean_doc(review.orginalText,False,False))
    create_output_aspect_polarity(review.labels, categories,list_output_train)



x_valid_segment = []
x_valid_nosegment = []
y_valid_category = []

output_val1 = []
output_val2 = []
output_val3 = []
output_val4 = []
output_val5 = []
output_val6 = []
output_val7 = []


list_output_val = list()
list_output_val.append(output_val1)
list_output_val.append(output_val2)
list_output_val.append(output_val3)
list_output_val.append(output_val4)
list_output_val.append(output_val5)
list_output_val.append(output_val6)
list_output_val.append(output_val7)


for review in read_data_dev():
    x_valid_segment.append(clean_doc(review.orginalText,True,False))
    x_valid_nosegment.append(clean_doc(review.orginalText,False,False))
    create_output_aspect_polarity(review.labels, categories,list_output_val)



x_test_segment = []
x_test_nosegment = []
x_test_original = []
for review in read_data_test():
    x_test_segment.append(clean_doc(review.orginalText,True,False))
    x_test_nosegment.append(clean_doc(review.orginalText,False,False))
    x_test_original.append(review.orginalText)


6522
976
1821


In [ ]:
with open(path_test,"r",encoding="utf8") as file:
  content = file.read()
  with open("grouth_true.txt","w",encoding="utf8") as file:
      file.write(content)

In [ ]:
print(len(x_train_segment),len(x_train_nosegment),len(list_output_train))
print(len(x_test_segment),len(x_test_nosegment))
print(len(x_valid_segment),len(x_valid_nosegment),len(list_output_val))

6522 6522 7
1821 1821
976 976 7


In [ ]:
print(x_train_segment[0])
print(x_train_nosegment[0])
print(x_train_segment[220])
print(x_train_nosegment[220])
print(x_train_segment[100])
print(x_train_nosegment[100])
print(x_train_segment[-1])
print(x_train_nosegment[-1])

cả nhà mình đi ăn hết hơn triệu piza với pasta cũng được ko có gì đặc_biệt lắm món salad khá ngon chua_chua ngọt ngọt dễ ăn gọi mấy món mà mình chỉ thích mỗi món salad này không_gian rộng_rãi thoải_mái nhưng_mà nhân_viên phục_vụ không nhiệt_tình bàn mình bị thiếu dĩa với thìa gọi mãi không thấy đưa ra chán nên khách tự đi lấy luôn
cả nhà mình đi ăn hết hơn triệu piza với pasta cũng được ko có gì đặc biệt lắm món salad khá ngon chua chua ngọt ngọt dễ ăn gọi mấy món mà mình chỉ thích mỗi món salad này không gian rộng rãi thoải mái nhưng mà nhân viên phục vụ không nhiệt tình bàn mình bị thiếu dĩa với thìa gọi mãi không thấy đưa ra chán nên khách tự đi lấy luôn
nhân_viên nhiệt_tình thức_ăn ngon nhưng hơi đắt nếu đi vào buổi trưa sẽ có set rẻ hơn buổi tối đông có_khi hết bàn phải chờ lâu
nhân viên nhiệt tình thức ăn ngon nhưng hơi đắt nếu đi vào buổi trưa sẽ có set rẻ hơn buổi tối đông có khi hết bàn phải chờ lâu
thấy deal này giá khá rẻ giá chỉ giá_tiền nên đi ăn thử xem sao quán nằm trên 

In [ ]:
# XLM-Align
model_name = 'microsoft/xlm-align-base'
config = AutoConfig.from_pretrained(model_name, output_hidden_states=True)
transformer_model = TFAutoModel.from_pretrained(model_name, config = config,from_pt=True)

# Train
tokenizer_model = AutoTokenizer.from_pretrained(model_name)
encoding_train = tokenizer_model(x_train_nosegment, return_tensors='tf', padding=True, truncation=True)
train_ids = encoding_train['input_ids']
train_attention_mask = encoding_train['attention_mask']
max_len_train = len(train_ids[0])
print("max length Train: ", max_len_train)

# Valid
encoding_valid = tokenizer_model(x_valid_nosegment, return_tensors='tf', padding="max_length", truncation=True, max_length=max_len_train)
valid_ids = encoding_valid['input_ids']
valid_attention_mask = encoding_valid['attention_mask']
valid_max_len = len(valid_ids[0])
print("max length Valid: ", valid_max_len)

# Test
encoding_test = tokenizer_model(x_test_nosegment, return_tensors='tf', padding="max_length", truncation=True, max_length=max_len_train)
test_ids = encoding_test['input_ids']
test_attention_mask = encoding_test['attention_mask']
test_max_len = len(test_ids[0])
print("max length test: ", test_max_len)

max length Train:  263
max length Valid:  263
max length test:  263


In [ ]:
# XLM-Align model
input_id = tf.keras.layers.Input(shape=(max_len_train,), name='input_token', dtype='int32')
input_mask = tf.keras.layers.Input(shape=(max_len_train,), name='mask_token', dtype='int32')

hidden_states = transformer_model(input_id,attention_mask = input_mask)[0]

cls_token = hidden_states[:,0,:]
cls_token = tf.keras.layers.Dropout(0.25)(cls_token)

output1 = tf.keras.layers.Dense(6,name="output1", activation='softmax')(cls_token)
output2 = tf.keras.layers.Dense(6,name="output2", activation='softmax')(cls_token)
output3 = tf.keras.layers.Dense(6,name="output3", activation='softmax')(cls_token)
output4 = tf.keras.layers.Dense(6,name="output4", activation='softmax')(cls_token)
output5 = tf.keras.layers.Dense(6,name="output5", activation='softmax')(cls_token)
output6 = tf.keras.layers.Dense(6,name="output6", activation='softmax')(cls_token)
output7 = tf.keras.layers.Dense(6,name="output7", activation='softmax')(cls_token)


model_xlmalign = tf.keras.Model(inputs=[input_id,input_mask], outputs=[output1,output2,output3,output4,output5,output6,output7])

opt = tf.keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.CategoricalCrossentropy()
metric = tf.metrics.CategoricalAccuracy('accuracy')

model_xlmalign.compile(loss={'output1':loss,'output2':loss,'output3':loss,'output4':loss,'output5':loss,'output6':loss,'output7':loss},optimizer=opt, metrics = [metric])

list_total_Y_train  = [np.array(list_output_train[0]),np.array(list_output_train[1]),np.array(list_output_train[2]),np.array(list_output_train[3]),np.array(list_output_train[4]),np.array(list_output_train[5]),np.array(list_output_train[6])]
list_total_Y_val  = [np.array(list_output_val[0]),np.array(list_output_val[1]),np.array(list_output_val[2]),np.array(list_output_val[3]),np.array(list_output_val[4]),np.array(list_output_val[5]),np.array(list_output_val[6])]

history = model_xlmalign.fit([train_ids,train_attention_mask], list_total_Y_train, validation_data=([valid_ids,valid_attention_mask],list_total_Y_val) ,batch_size=16, epochs=50)
print(model_xlmalign.summary())

Epoch 1/50


408/408 [==============================] - 453s 1s/step - loss: 8.9497 - output1_loss: 1.4615 - output2_loss: 0.8595 - output3_loss: 1.5449 - output4_loss: 1.3529 - output5_loss: 1.4640 - output6_loss: 1.2632 - output7_loss: 1.0037 - output1_accuracy: 0.4589 - output2_accuracy: 0.7870 - output3_accuracy: 0.3855 - output4_accuracy: 0.4108 - output5_accuracy: 0.3559 - output6_accuracy: 0.5276 - output7_accuracy: 0.7246 - val_loss: 8.4530 - val_output1_loss: 1.3829 - val_output2_loss: 0.7909 - val_output3_loss: 1.4609 - val_output4_loss: 1.3148 - val_output5_loss: 1.4057 - val_output6_loss: 1.1746 - val_output7_loss: 0.9232 - val_output1_accuracy: 0.4785 - val_output2_accuracy: 0.7920 - val_output3_accuracy: 0.4191 - val_output4_accuracy: 0.3463 - val_output5_accuracy: 0.3299 - val_output6_accuracy: 0.5584 - val_output7_accuracy: 0.7295
Epoch 2/50
408/408 [==============================] - 383s 938ms/step - loss: 8.5017 - output1_loss: 1.4054 - output2_loss: 0.8077 - output3_loss: 1.4921 

In [ ]:
textPrint =  ""
count = 0
for index,item in enumerate(test_ids):
    predicted = model_xlmalign.predict([np.expand_dims(item, axis=0),np.expand_dims(np.array(test_attention_mask[index]), axis=0)],verbose=0)
    s = ''
    for i, predict in enumerate(predicted):
        index2, value = max(enumerate(predict[0]), key=operator.itemgetter(1))
        if index2 == 1:
              s+= '{' + str(categories[i]) + ', very_positive}, '
        elif index2 == 2:
            s+= '{' + str(categories[i]) + ', positive}, '
        elif index2 == 3:
            s+= '{' + str(categories[i]) + ', neutral}, '
        elif index2 == 4:
            s+= '{' + str(categories[i]) + ', negative}, '
        elif index2 == 5:
            s+= '{' + str(categories[i]) + ', very_negative}, '
    if s.strip() == "":
      s = "{MISCELLANEOUS, neutral}, "
    textPrint += '#' + str(count +1)+'\n'
    textPrint += x_test_original[index] + '\n'
    textPrint += s[:len(s)-2] +'\n\n'
    count +=1
textPrint = textPrint[:-2]
with open('output_XLMAlign.txt','w',encoding = 'utf8') as file:
    file.write(textPrint)
print("Done")

!python EvaluationSystemByFile.py "grouth_true.txt" "output_XLMAlign.txt"

Done
-------------------------------------------------------------
-------------------------------------------------------------
Mean Precision score:  71.08
Mean Recall score:  71.64
Mean F1 score:  71.36
-------------------------------------------------------------
-------------------------------------------------------------


In [ ]:
# PhoBERT base

from transformers import TFAutoModel, AutoTokenizer

tokenizer_phobert = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)

transformer_model_phobert = TFAutoModel.from_pretrained("vinai/phobert-base")

encoding_phobert = tokenizer_phobert(x_train_segment, return_tensors='tf', padding=True, truncation=True)
input_ids_phobert = encoding_phobert['input_ids']
attention_mask_phobert = encoding_phobert['attention_mask']
max_len_phobert = len(input_ids_phobert[0])
print("max length PhoBERT: ", max_len_phobert)

# Valid
encoding_phobert = tokenizer_phobert(x_valid_segment, return_tensors='tf', padding="max_length", truncation=True, max_length=max_len_phobert)
valid_ids_phobert = encoding_phobert['input_ids']
valid_attention_mask_phobert = encoding_phobert['attention_mask']
valid_max_len_phobert = len(valid_ids_phobert[0])
print("max length PhoBERT: ", valid_max_len_phobert)

# Test
encoding_phobert = tokenizer_phobert(x_test_segment, return_tensors='tf', padding="max_length", truncation=True, max_length=max_len_phobert)
test_ids_phobert = encoding_phobert['input_ids']
test_attention_mask_phobert = encoding_phobert['attention_mask']
test_max_len_phobert = len(test_ids_phobert[0])
print("max length PhoBERT: ", test_max_len_phobert)

max length PhoBERT:  256
max length PhoBERT:  256
max length PhoBERT:  256


In [ ]:
# PhoBERT model

input_ids =  tf.keras.layers.Input(shape=(max_len_phobert,), name='input_token_phobert', dtype='int32')
input_masks =  tf.keras.layers.Input(shape=(max_len_phobert,), name='mask_token_phobert', dtype='int32')

hidden_states = transformer_model_phobert(input_ids,attention_mask = input_masks)[0]

cls_token = hidden_states[:,0,:]

cls_token = tf.keras.layers.Dropout(0.25)(cls_token)

output1 = tf.keras.layers.Dense(6,name="output1", activation='softmax')(cls_token)
output2 = tf.keras.layers.Dense(6,name="output2", activation='softmax')(cls_token)
output3 = tf.keras.layers.Dense(6,name="output3", activation='softmax')(cls_token)
output4 = tf.keras.layers.Dense(6,name="output4", activation='softmax')(cls_token)
output5 = tf.keras.layers.Dense(6,name="output5", activation='softmax')(cls_token)
output6 = tf.keras.layers.Dense(6,name="output6", activation='softmax')(cls_token)
output7 = tf.keras.layers.Dense(6,name="output7", activation='softmax')(cls_token)


model_phobert = tf.keras.Model(inputs=[input_ids,input_masks], outputs=[output1,output2,output3,output4,output5,output6,output7])

opt = tf.keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.CategoricalCrossentropy()
metric = tf.metrics.CategoricalAccuracy('accuracy')

model_phobert.compile(loss={'output1':loss,'output2':loss,'output3':loss,'output4':loss,'output5':loss,'output6':loss,'output7':loss},optimizer=opt, metrics = [metric])

list_total_Y_train  = [np.array(list_output_train[0]),np.array(list_output_train[1]),np.array(list_output_train[2]),np.array(list_output_train[3]),np.array(list_output_train[4]),np.array(list_output_train[5]),np.array(list_output_train[6])]
list_total_Y_val  = [np.array(list_output_val[0]),np.array(list_output_val[1]),np.array(list_output_val[2]),np.array(list_output_val[3]),np.array(list_output_val[4]),np.array(list_output_val[5]),np.array(list_output_val[6])]

history = model_phobert.fit([input_ids_phobert,attention_mask_phobert], list_total_Y_train, validation_data=([valid_ids_phobert,valid_attention_mask_phobert],list_total_Y_val) ,batch_size=8, epochs=50)
print(model_phobert.summary())

Epoch 1/50


816/816 [==============================] - 450s 503ms/step - loss: 6.4222 - output1_loss: 1.1334 - output2_loss: 0.6163 - output3_loss: 1.0101 - output4_loss: 0.8892 - output5_loss: 0.8876 - output6_loss: 1.0313 - output7_loss: 0.8543 - output1_accuracy: 0.5529 - output2_accuracy: 0.8128 - output3_accuracy: 0.6150 - output4_accuracy: 0.6887 - output5_accuracy: 0.6891 - output6_accuracy: 0.6089 - output7_accuracy: 0.7314 - val_loss: 4.5268 - val_output1_loss: 0.8802 - val_output2_loss: 0.3903 - val_output3_loss: 0.5471 - val_output4_loss: 0.6314 - val_output5_loss: 0.5865 - val_output6_loss: 0.7649 - val_output7_loss: 0.7265 - val_output1_accuracy: 0.6588 - val_output2_accuracy: 0.8668 - val_output3_accuracy: 0.8115 - val_output4_accuracy: 0.7736 - val_output5_accuracy: 0.8012 - val_output6_accuracy: 0.7408 - val_output7_accuracy: 0.7459
Epoch 2/50
816/816 [==============================] - 377s 462ms/step - loss: 4.0071 - output1_loss: 0.7499 - output2_loss: 0.3859 - output3_loss: 0.52

In [ ]:
import operator
textPrint =  ""
count = 0
for index,item in enumerate(test_ids_phobert):
    predicted = model_phobert.predict([np.expand_dims(item, axis=0),np.expand_dims(np.array(test_attention_mask_phobert[index]), axis=0)],verbose=0)
    s = ''
    for i, predict in enumerate(predicted):
        index2, value = max(enumerate(predict[0]), key=operator.itemgetter(1))
        if index2 == 1:
              s+= '{' + str(categories[i]) + ', very_positive}, '
        elif index2 == 2:
            s+= '{' + str(categories[i]) + ', positive}, '
        elif index2 == 3:
            s+= '{' + str(categories[i]) + ', neutral}, '
        elif index2 == 4:
            s+= '{' + str(categories[i]) + ', negative}, '
        elif index2 == 5:
            s+= '{' + str(categories[i]) + ', very_negative}, '
    if s.strip() == "":
      s = "{MISCELLANEOUS, neutral}, "
    textPrint += '#' + str(count +1)+'\n'
    textPrint += x_test_original[index] + '\n'
    textPrint += s[:len(s)-2] +'\n\n'
    count +=1
textPrint = textPrint[:-2]
with open('output_PHOBERT.txt','w',encoding = 'utf8') as file:
    file.write(textPrint)
print("Done")

Done


In [ ]:
# Caculate score of phoBERT
print("Score of PhoBERT model")
!python EvaluationSystemByFile.py "grouth_true.txt" "output_PHOBERT.txt"

In [ ]:
# Caculate score of phoBERT
print("Score of PhoBERT model")
!python EvaluationSystemByFile.py "grouth_true.txt" "output_PHOBERT.txt"

Score of PhoBERT model
-------------------------------------------------------------
-------------------------------------------------------------
Mean Precision score:  72.43
Mean Recall score:  72.69
Mean F1 score:  72.56
-------------------------------------------------------------
-------------------------------------------------------------


In [ ]:
from transformers import XLMRobertaConfig, TFXLMRobertaModel, XLMRobertaTokenizerFast

model_name = 'xlm-roberta-base'

config = XLMRobertaConfig.from_pretrained(model_name, output_hidden_states=True)

transformer_model_xlmr = TFXLMRobertaModel.from_pretrained(model_name, config = config,from_pt=True)

# Train XLMR
tokenizer_xlmr = XLMRobertaTokenizerFast.from_pretrained(model_name, do_lower_case=False)
encoding_xlmr = tokenizer_xlmr(x_train_nosegment, return_tensors='tf', padding=True, truncation=True)
input_ids_xlmr = encoding_xlmr['input_ids']
attention_mask_xlmr = encoding_xlmr['attention_mask']
max_len_xlmr = len(input_ids_xlmr[0])
print("max length XLM-R: ", max_len_xlmr)

# Valid for XLM-R
encoding_xlmr = tokenizer_xlmr(x_valid_nosegment, return_tensors='tf', padding="max_length", truncation=True, max_length=max_len_xlmr)
valid_ids_xlmr = encoding_xlmr['input_ids']
valid_attention_mask_xlmr = encoding_xlmr['attention_mask']
valid_max_len_xlmr = len(valid_ids_xlmr[0])
print("max length XLM-R: ", valid_max_len_xlmr)

# Test for XLM-R
encoding_xlmr = tokenizer_xlmr(x_test_nosegment, return_tensors='tf', padding="max_length", truncation=True, max_length=max_len_xlmr)
test_ids_xlmr = encoding_xlmr['input_ids']
test_attention_mask_xlmr = encoding_xlmr['attention_mask']
test_max_len_xlmr = len(test_ids_xlmr[0])
print("max length XLM-R: ", test_max_len_xlmr)

max length XLM-R:  263
max length XLM-R:  263
max length XLM-R:  263


In [ ]:
# XLM-R model
input_id_xmlr = tf.keras.layers.Input(shape=(max_len_xlmr,), name='input_token_xlmr', dtype='int32')
input_mask_xlmr = tf.keras.layers.Input(shape=(max_len_xlmr,), name='mask_token_xlmr', dtype='int32')

hidden_states_xlmr = transformer_model_xlmr(input_id_xmlr,attention_mask = input_mask_xlmr)[0]

cls_token = hidden_states_xlmr[:,0,:]
cls_token = tf.keras.layers.Dropout(0.25)(cls_token)

output1 = tf.keras.layers.Dense(6,name="output1", activation='softmax')(cls_token)
output2 = tf.keras.layers.Dense(6,name="output2", activation='softmax')(cls_token)
output3 = tf.keras.layers.Dense(6,name="output3", activation='softmax')(cls_token)
output4 = tf.keras.layers.Dense(6,name="output4", activation='softmax')(cls_token)
output5 = tf.keras.layers.Dense(6,name="output5", activation='softmax')(cls_token)
output6 = tf.keras.layers.Dense(6,name="output6", activation='softmax')(cls_token)
output7 = tf.keras.layers.Dense(6,name="output7", activation='softmax')(cls_token)

model_xlmr = tf.keras.Model(inputs=[input_id_xmlr,input_mask_xlmr], outputs=[output1,output2,output3,output4,output5,output6,output7])

opt = tf.keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.CategoricalCrossentropy()
metric = tf.metrics.CategoricalAccuracy('accuracy')

model_xlmr.compile(loss={'output1':loss,'output2':loss,'output3':loss,'output4':loss,'output5':loss,'output6':loss,'output7':loss},optimizer=opt, metrics = [metric])

list_total_Y_train  = [np.array(list_output_train[0]),np.array(list_output_train[1]),np.array(list_output_train[2]),np.array(list_output_train[3]),np.array(list_output_train[4]),np.array(list_output_train[5]),np.array(list_output_train[6])]
list_total_Y_val  = [np.array(list_output_val[0]),np.array(list_output_val[1]),np.array(list_output_val[2]),np.array(list_output_val[3]),np.array(list_output_val[4]),np.array(list_output_val[5]),np.array(list_output_val[6])]

history = model_xlmr.fit([input_ids_xlmr,attention_mask_xlmr], list_total_Y_train, validation_data=([valid_ids_xlmr,valid_attention_mask_xlmr],list_total_Y_val) ,batch_size=8, epochs=50)
print(model_xlmr.summary())


In [ ]:
textPrint =  ""
count = 0
for index,item in enumerate(test_ids_xlmr):
    predicted = model_xlmr.predict([np.expand_dims(item, axis=0),np.expand_dims(np.array(test_attention_mask_xlmr[index]), axis=0)],verbose=0)
    s = ''
    for i, predict in enumerate(predicted):
        index2, value = max(enumerate(predict[0]), key=operator.itemgetter(1))
        if index2 == 1:
              s+= '{' + str(categories[i]) + ', very_positive}, '
        elif index2 == 2:
            s+= '{' + str(categories[i]) + ', positive}, '
        elif index2 == 3:
            s+= '{' + str(categories[i]) + ', neutral}, '
        elif index2 == 4:
            s+= '{' + str(categories[i]) + ', negative}, '
        elif index2 == 5:
            s+= '{' + str(categories[i]) + ', very_negative}, '
    if s.strip() == "":
      s = "{MISCELLANEOUS, neutral}, "
    textPrint += '#' + str(count +1)+'\n'
    textPrint += x_test_original[index] + '\n'
    textPrint += s[:len(s)-2] +'\n\n'
    count +=1
textPrint = textPrint[:-2]
with open('output_XLMR.txt','w',encoding = 'utf8') as file:
    file.write(textPrint)
print("Done")

!python EvaluationSystemByFile.py "grouth_true.txt" "output_XLMR.txt"